In [17]:
import sys
import os

def get_UGCE_directory():
    """Get the path of the 'UGCE-User-Guided-Counterfactual-Exploration' directory."""
    current_dir = os.getcwd()
    target_dir = 'UGCE-User-Guided-Counterfactual-Exploration'
    
    while os.path.basename(current_dir) != target_dir:
        current_dir = os.path.dirname(current_dir)
        if current_dir == os.path.dirname(current_dir):
            return None
        
    return current_dir

def get_system_slash():
    """Get the system-specific directory separator."""
    return os.sep

UGCE_dir = get_UGCE_directory()
sys.path.append(UGCE_dir)
sep = get_system_slash()
sys.path.append(UGCE_dir + get_system_slash() + 'src')

from dataLoader import *
from utils import *

In [18]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
seed_number = 42
import random

random.seed(seed_number)
np.random.seed(seed_number)

In [20]:
datasetName = 'AdultCA'

In [21]:
from folktables import ACSDataSource, ACSIncome

# Initialize the data source for California in 2024
data_source = ACSDataSource(survey_year='2023', horizon='1-Year', survey='person')

# Download and extract the data for California
ca_data = data_source.get_data(states=['CA'], download=True)

# Define the ACSIncome task
features, label, group = ACSIncome.df_to_numpy(ca_data)
feature_names = ACSIncome.features

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import pandas as pd
import dice_ml
from dice_ml.utils import helpers

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

dataset = pd.DataFrame(features, columns=feature_names)
TARGET_COLUMN = 'target'
dataset['target'] = label
dataset['target'] = LabelEncoder().fit_transform(dataset['target'])
target = dataset['target']
datasetX = dataset.drop('target', axis=1)

x_train, x_test, y_train, y_test = train_test_split(datasetX,
                                                    target,
                                                    test_size=0.2,
                                                    random_state=0,
                                                    stratify=target)

numerical = datasetX.columns.to_list()
categorical = x_train.columns.difference(numerical)

try:
    import joblib
    model = joblib.load(f"{ugce_dir}/results/models/{datasetName}_model.pkl")
except:
    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())])

    categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    transformations = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical),
            ('cat', categorical_transformer, categorical)])

    model = RandomForestClassifier(random_state=42)

    model = Pipeline(steps=[('preprocessor', transformations),
                        ('classifier', model)])

    model.fit(x_train, y_train)

    import joblib
    os.makedirs(f"{ugce_dir}/results/models", exist_ok=True)
    joblib.dump(model, f"{ugce_dir}/results/models/{datasetName}_model.pkl")

y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

negative_instances = x_test[model.predict(x_test) == 0]
instances_to_explain = negative_instances

Accuracy:  0.8063754427390791


In [22]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

In [23]:
numerical_columns = iea.dataset.select_dtypes(include=['int64', 'float64']).columns

non_zero_descriptions = {}

for col in numerical_columns:
    non_zero_values = iea.dataset[iea.dataset[col] != 0][col]
    if not non_zero_values.empty:
        non_zero_descriptions[col] = non_zero_values.describe()

# Display results
for feature, stats in non_zero_descriptions.items():
    print(f"\n Feature: {feature}")
    print(stats)


 Feature: AGEP
count    203278.000000
mean         43.085095
std          15.109688
min          17.000000
25%          31.000000
50%          42.000000
75%          55.000000
max          94.000000
Name: AGEP, dtype: float64

 Feature: COW
count    203278.000000
mean          2.183158
std           1.885850
min           1.000000
25%           1.000000
50%           1.000000
75%           3.000000
max           8.000000
Name: COW, dtype: float64

 Feature: SCHL
count    203278.000000
mean         18.506587
std           4.180591
min           1.000000
25%          16.000000
50%          19.000000
75%          21.000000
max          24.000000
Name: SCHL, dtype: float64

 Feature: MAR
count    203278.000000
mean          2.732293
std           1.867056
min           1.000000
25%           1.000000
50%           1.000000
75%           5.000000
max           5.000000
Name: MAR, dtype: float64

 Feature: OCCP
count    203278.000000
mean       3872.819641
std        2676.274145
min        

In [10]:
iea.numerical_columns

['AGEP', 'COW', 'SCHL', 'MAR', 'OCCP', 'POBP', 'WKHP', 'SEX', 'RAC1P']

# Constraints Type Series:
1. Immutability
2. Ranges
3. Directionality

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    1: {
        'SEX': 'i',
        'RAC1P': 'i'
    },
    2: {
        'SEX': 'i',
        'RAC1P': 'i',  
        'WKHP': (32, 70),
        'OCCP': (10, 3000)
    },
    3: {
        'SEX': 'i',
        'RAC1P': 'i',  
        'WKHP': (32, 70),
        'OCCP': (10, 3000),
        'AGEP': 'incr',
        'SCHL': 'incr'
    }
}
for key in list(updated_constraints.keys()):
    values = updated_constraints[key]
    for col in iea.feature_names:
        if col not in values:
            updated_constraints[key][col] = ''

results_incremental_imm_ranges_direct_arr = []
import time
strategy = "fix_population_update_fitness"
for i in range(5):
    results_incremental_imm_ranges_direct = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.9, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=True,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_imm_ranges_direct_arr.append(results_incremental_imm_ranges_direct)
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_imm_ranges_direct_arr, open(f"{results_dir}/results_incremental_imm_ranges_direct_arr.pkl", "wb"))

100%|██████████| 2063/2063 [14:37<00:00,  2.35it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [15:05<00:00,  2.28it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [15:41<00:00,  2.19it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [15:16<00:00,  2.25it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [14:42<00:00,  2.34it/s]


Empty intermediate counter: 0


In [8]:
## load the results
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_imm_ranges_direct_arr = pickle.load(open(f'{results_dir}/results_incremental_imm_ranges_direct_arr.pkl', 'rb'))

# Constraints Type Series:
1. Ranges
2. Immutability
3. Directionality

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    1: {
        'WKHP': (32, 70),
        'OCCP': (10, 3000)
    },
    2: {
        'WKHP': (32, 70),
        'OCCP': (10, 3000),
        'SEX': 'i',
        'RAC1P': 'i'
    },
    3: {
        'WKHP': (32, 70),
        'OCCP': (10, 3000),
        'SEX': 'i',
        'RAC1P': 'i',
        'AGEP': 'incr',
        'SCHL': 'incr'
    }
}
for key in list(updated_constraints.keys()):
    values = updated_constraints[key]
    for col in iea.feature_names:
        if col not in values:
            updated_constraints[key][col] = ''

results_incremental_ranges_imm_incr_arr = []
import time
strategy = "fix_population_update_fitness"
for i in range(5):
    results_incremental_ranges_imm_incr = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.9, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=True,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_ranges_imm_incr_arr.append(results_incremental_ranges_imm_incr)
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_ranges_imm_incr_arr, open(f"{results_dir}/results_incremental_ranges_imm_incr_arr.pkl", "wb"))

100%|██████████| 2063/2063 [14:32<00:00,  2.37it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [13:48<00:00,  2.49it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [14:30<00:00,  2.37it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [14:35<00:00,  2.36it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [14:40<00:00,  2.34it/s]


Empty intermediate counter: 0


In [14]:
## load the results
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_ranges_imm_incr_arr = pickle.load(open(f'{results_dir}/results_incremental_ranges_imm_incr_arr.pkl', 'rb'))

# Constraints Type Series:
1. Directionality
2. Immutability
3. Ranges

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    1: {
        'AGEP': 'incr',
        'SCHL': 'incr'
    },
    2: {
        'AGEP': 'incr',
        'SCHL': 'incr',
        'SEX': 'i',
        'RAC1P': 'i'
    },
    3: {
        'AGEP': 'incr',
        'SCHL': 'incr',
        'SEX': 'i',
        'RAC1P': 'i',
        'WKHP': (32, 70),
        'OCCP': (10, 3000),
    }
}
for key in list(updated_constraints.keys()):
    values = updated_constraints[key]
    for col in iea.feature_names:
        if col not in values:
            updated_constraints[key][col] = ''

import time
strategy = "fix_population_update_fitness"
results_incremental_dir_im_range_arr = []
for i in range(1):
    results_incremental_dir_im_range = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.9, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=True,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_dir_im_range_arr.append(results_incremental_dir_im_range)
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_dir_im_range_arr, open(f"{results_dir}/results_incremental_dir_im_range_arr.pkl", "wb"))

In [13]:
## load the results
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_dir_im_range_arr = pickle.load(open(f'{results_dir}/results_incremental_dir_im_range_arr.pkl', 'rb'))

# Make Plots

In [ ]:
from test_utils import gather_results_sequence_of_type_constraints
import matplotlib.pyplot as plt

constraint_orders = ["I→R→D", "R→I→D", "D→I→R"]

time_dynamic_imm_ranges_direct, avg_generations_imm_ranges_direct, avg_cfes_found_imm_ranges_direct, avg_proximity_loss_imm_ranges_direct, avg_sparsity_imm_ranges_direct, avg_intermediate_imm_ranges_incr, \
time_dynamic_ranges_imm_incr, avg_generations_ranges_imm_incr, avg_cfes_found_ranges_imm_incr, avg_proximity_loss_ranges_imm_incr, avg_sparsity_ranges_imm_incr, avg_intermediate_ranges_imm_incr, \
time_dynamic_dir_im_range, avg_generations_dir_im_range, avg_cfes_found_dir_im_range, avg_proximity_loss_dir_im_range, avg_sparsity_dir_im_range, avg_intermediate_dir_im_range =\
    gather_results_sequence_of_type_constraints(iea, results_incremental_imm_ranges_direct_arr, results_incremental_ranges_imm_incr_arr, results_incremental_dir_im_range, verbose=True) 

cfe_found = [
        avg_cfes_found_imm_ranges_direct,
        avg_cfes_found_ranges_imm_incr,
        avg_cfes_found_dir_im_range
]
avg_time = [
    time_dynamic_imm_ranges_direct,
    time_dynamic_ranges_imm_incr,
    time_dynamic_dir_im_range
]
avg_weighted_l1 = [
    avg_proximity_loss_imm_ranges_direct,
    avg_proximity_loss_ranges_imm_incr,
    avg_proximity_loss_dir_im_range
]

avg_sparsity = [
    avg_sparsity_imm_ranges_direct,
    avg_sparsity_ranges_imm_incr,
    avg_sparsity_dir_im_range
]
results = {
    "cfe_found": cfe_found,
    "avg_time": avg_time,
    "avg_weighted_l1": avg_weighted_l1,
    "avg_sparsity": avg_sparsity
}

In [16]:
table_data = pd.DataFrame(results, index=constraint_orders)
print(table_data.to_latex(float_format="%.4f"))

\begin{tabular}{lrrrr}
\toprule
 & cfe_found & avg_time & avg_weighted_l1 & avg_sparsity \\
\midrule
I→R→D & 99.9901 & 144.7059 & 0.0642 & 0.0538 \\
R→I→D & 99.9901 & 170.4225 & 0.0642 & 0.0538 \\
D→I→R & 99.9901 & 173.5455 & 0.1020 & 0.0644 \\
\bottomrule
\end{tabular}

